In [3]:
from pathlib import Path
import os
import numpy as np

from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from google import genai

In [4]:
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found. Check your .env file."
    )

print("Gemini API key loaded successfully!")

Gemini API key loaded successfully!


In [5]:
client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Gemini client created successfully!")

Gemini client created successfully!


In [6]:
response = client.interactions.create(
    model="gemini-3.5-flash-lite",
    input="What is machine learning?"
)

print(response.output_text)

At its core, **machine learning (ML)** is a branch of artificial intelligence (AI) that allows computers to learn from data and improve at tasks without being explicitly programmed for every single step. 

To understand it, it helps to look at the difference between traditional programming and machine learning:

*   **Traditional Programming:** You give a computer **rules** and **data**, and it gives you **answers**. *(Example: You write rules for what a cat looks like, feed in photos, and the computer tells you if it's a cat).*
*   **Machine Learning:** You give a computer **data** and the correct **answers**, and it figures out the **rules**. *(Example: You feed the computer thousands of photos of cats and dogs labeled as such, and the computer figures out the rules that separate cats from dogs on its own).*

---

### How Does It Work?
1.  **Data Collection:** You feed a massive amount of data into the system (e.g., past weather data, customer purchase history, or images of traffic s

In [7]:
documents = []

for file_path in sorted(Path("docs").glob("*.txt")):
    text = file_path.read_text(
        encoding="utf-8"
    )

    documents.append({
        "filename": file_path.name,
        "text": text
    })

print("Documents loaded:", len(documents))

Documents loaded: 3


In [8]:
for document in documents:
    print("\nFile:", document["filename"])
    print("------------------------")
    print(document["text"])


File: dl.txt
------------------------
Deep learning is a subset of machine learning.
It uses artificial neural networks with multiple layers.
Neural networks contain input, hidden and output layers.
CNNs are commonly used for image classification.
RNNs can be used to process sequential data.
Deep learning is widely used in computer vision,
natural language processing and speech recognition.

File: ml.txt
------------------------
Machine learning is a branch of artificial intelligence.
It allows computers to learn patterns from data.
Supervised learning uses labelled training data.
Unsupervised learning finds patterns in unlabelled data.
Common machine learning algorithms include Linear Regression,
Decision Tree, Random Forest and Logistic Regression.
Machine learning is used for prediction and classification.
The main purpose of machine learning is to learn from data
and make predictions or decisions without being explicitly programmed.

File: python.txt
------------------------
Pytho

In [9]:
def split_text(text, chunk_size=20):

    words = text.split()

    chunks = []

    for i in range(0, len(words), chunk_size):

        chunk = " ".join(
            words[i:i + chunk_size]
        )

        chunks.append(chunk)

    return chunks

In [10]:
chunks = []

for document in documents:

    document_chunks = split_text(
        document["text"],
        chunk_size=20
    )

    for chunk in document_chunks:

        chunks.append({
            "filename": document["filename"],
            "text": chunk
        })

print("Total chunks:", len(chunks))

Total chunks: 10


In [11]:
for i, chunk in enumerate(chunks):

    print(f"\nChunk {i + 1}")
    print("Source:", chunk["filename"])
    print("------------------------")
    print(chunk["text"])


Chunk 1
Source: dl.txt
------------------------
Deep learning is a subset of machine learning. It uses artificial neural networks with multiple layers. Neural networks contain input,

Chunk 2
Source: dl.txt
------------------------
hidden and output layers. CNNs are commonly used for image classification. RNNs can be used to process sequential data. Deep

Chunk 3
Source: dl.txt
------------------------
learning is widely used in computer vision, natural language processing and speech recognition.

Chunk 4
Source: ml.txt
------------------------
Machine learning is a branch of artificial intelligence. It allows computers to learn patterns from data. Supervised learning uses labelled

Chunk 5
Source: ml.txt
------------------------
training data. Unsupervised learning finds patterns in unlabelled data. Common machine learning algorithms include Linear Regression, Decision Tree, Random Forest

Chunk 6
Source: ml.txt
------------------------
and Logistic Regression. Machine learning is us

In [12]:
def split_text(text, chunk_size=20):

    words = text.split()

    chunks = []

    for i in range(0, len(words), chunk_size):

        chunk = " ".join(
            words[i:i + chunk_size]
        )

        chunks.append(chunk)

    return chunks

In [13]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1599.18it/s]


Embedding model loaded successfully!


In [14]:
def get_embedding(text):

    embedding = embedding_model.encode(text)

    return embedding

In [15]:
test_embedding = get_embedding(
    chunks[0]["text"]
)

print("Embedding created successfully.")
print("Number of values:", len(test_embedding))

Embedding created successfully.
Number of values: 384


In [16]:
doc_embeddings = np.array(
    [
        get_embedding(chunk["text"])
        for chunk in chunks
    ]
)

print("Embedding shape:", doc_embeddings.shape)

Embedding shape: (10, 384)


In [17]:
def search_documents(question):

    question_embedding = np.array(
        get_embedding(question)
    )

    similarities = (
        doc_embeddings @ question_embedding
        /
        (
            np.linalg.norm(
                doc_embeddings,
                axis=1
            )
            *
            np.linalg.norm(
                question_embedding
            )
        )
    )

    best_index = np.argmax(similarities)

    return (
        chunks[best_index],
        similarities[best_index],
        best_index
    )

In [18]:
question = "What is machine learning?"

relevant_chunk, score, index = search_documents(
    question
)

print("Question:")
print(question)

print("\nRelevant Chunk:")
print(relevant_chunk["text"])

print("\nSource:")
print(relevant_chunk["filename"])

print("\nSimilarity Score:")
print(round(float(score), 4))

Question:
What is machine learning?

Relevant Chunk:
Machine learning is a branch of artificial intelligence. It allows computers to learn patterns from data. Supervised learning uses labelled

Source:
ml.txt

Similarity Score:
0.7892


In [19]:
def ask_my_docs(question):

    # Step 1: Find relevant chunk
    relevant_chunk, score, index = search_documents(
        question
    )

    # Step 2: Get context
    context = relevant_chunk["text"]

    # Step 3: Create prompt
    prompt = f"""
You are a document question-answering assistant.

Answer the question using ONLY the provided context.

Context:
{context}

Question:
{question}

If the answer is not available in the context,
say "I could not find the answer in the documents."

Answer:
"""

    # Step 4: Send prompt to Gemini
    response = client.interactions.create(
        model="gemini-3.5-flash-lite",
        input=prompt
    )

    # Step 5: Get Gemini answer
    answer = response.output_text

    return (
        answer,
        relevant_chunk,
        score
    )

In [20]:
question = "What is machine learning?"

answer, source, score = ask_my_docs(
    question
)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

print("\nSource:")
print(source["filename"])

print("\nSimilarity Score:")
print(round(float(score), 4))

Question:
What is machine learning?

Answer:
Based on the provided context, machine learning is a branch of artificial intelligence that allows computers to learn patterns from data.

Source:
ml.txt

Similarity Score:
0.7892


In [21]:
question = "What is machine learning?"

answer, source, score = ask_my_docs(question)

print("Answer:")
print(answer)

print("\nSource:")
print(source["filename"])

Answer:
Based on the provided context, machine learning is a branch of artificial intelligence that allows computers to learn patterns from data.

Source:
ml.txt


In [22]:
question = "What is machine learning?"

answer, source, score = ask_my_docs(
    question
)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

print("\nSource:")
print(source["filename"])

print("\nSimilarity Score:")
print(round(float(score), 4))

Question:
What is machine learning?

Answer:
Based on the provided context, machine learning is a branch of artificial intelligence that allows computers to learn patterns from data.

Source:
ml.txt

Similarity Score:
0.7892


In [23]:
question = "What is Python used for?"

answer, source, score = ask_my_docs(question)

print("Answer:")
print(answer)

print("\nSource:")
print(source["filename"])

Answer:
Python is widely used in data science.

Source:
python.txt
